# **Análise da Série Histórica da Soja - CONAB**

**Série histórica nacional da produção de soja no Brasil, organizada por safra e por região ou estado.**

**Dados de área plantada, produtividade e produção.**

**área plantada, em mil hectares;
produtividade, normalmente em kg/ha;
produção, normalmente em mil toneladas.**

**Frequência: Anual por safra**

**A produção da CONAB representa a quantidade total produzida ou estimada. Ela não significa somente exportação. Essa produção pode ser destinada a:
exportação;
processamento industrial;
produção de óleo e farelo;
alimentação animal;
estoque;
consumo interno.**

**Esse arquivo representa a dimensão de oferta agrícola.**

In [14]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

In [15]:
arquivo = "/content/SojaSerieHist.xls"

excel = pd.ExcelFile(arquivo)

print("Abas encontradas:")
for aba in excel.sheet_names:
    print("-", aba)

Abas encontradas:
- Área
- Produtividade
- Produção


In [16]:
def normalizar_texto(texto):
    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    return texto.lower().strip()


def carregar_aba(arquivo, aba):
    # Carrega inicialmente sem cabeçalho
    bruto = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=None
    )

    # Procura a linha que contém o cabeçalho
    linha_cabecalho = None

    for indice, linha in bruto.head(20).iterrows():
        texto_linha = " ".join(
            normalizar_texto(valor)
            for valor in linha.dropna()
        )

        if "regiao" in texto_linha or "uf" in texto_linha:
            linha_cabecalho = indice
            break

    # Caso não encontre, utiliza a linha com mais valores preenchidos
    if linha_cabecalho is None:
        linha_cabecalho = bruto.head(20).notna().sum(axis=1).idxmax()

    # Recarrega usando o cabeçalho correto
    df = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=linha_cabecalho
    )

    # Remove linhas e colunas completamente vazias
    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    # Arruma os nomes das colunas
    df.columns = [
        str(coluna).strip()
        if not str(coluna).startswith("Unnamed")
        else f"coluna_{numero}"
        for numero, coluna in enumerate(df.columns)
    ]

    # Renomeia a primeira coluna
    df = df.rename(columns={df.columns[0]: "Localidade"})

    # Remove linhas sem localidade
    df = df.dropna(subset=["Localidade"])

    # Remove espaços dos nomes
    df["Localidade"] = df["Localidade"].astype(str).str.strip()

    return df.reset_index(drop=True)

tabelas = {}

for aba in excel.sheet_names:
    tabelas[aba] = carregar_aba(arquivo, aba)
    print(f"Aba carregada: {aba} — {tabelas[aba].shape}")

Aba carregada: Área — (37, 51)
Aba carregada: Produtividade — (37, 51)
Aba carregada: Produção — (37, 51)


In [17]:
area = None
produtividade = None
producao = None

for nome_aba, tabela in tabelas.items():
    nome_normalizado = normalizar_texto(nome_aba)

    if "area" in nome_normalizado:
        area = tabela

    elif "produtiv" in nome_normalizado:
        produtividade = tabela

    elif "producao" in nome_normalizado:
        producao = tabela

print("Área encontrada:", area is not None)
print("Produtividade encontrada:", produtividade is not None)
print("Produção encontrada:", producao is not None)

Área encontrada: True
Produtividade encontrada: True
Produção encontrada: True


In [18]:
display(area.head(15))

,Localidade,1976/77,1977/78,1978/79,1979/80,1980/81,1981/82,1982/83,1983/84,1984/85,...,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26 Previsão (¹)
0,NORTE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1809.0,1931.7,1988.3,2110.8,2424.6,2773.1,3122.245,3394.9,3697.5,4021.0
1,RR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,30.0,38.2,40.0,49.8,70.0,95.0,123.000,123.0,135.9,145.5
2,RO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,296.0,333.6,333.7,348.4,396.5,491.7,595.000,643.2,694.7,719.7
3,AC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.5,1.5,4.0,6.1,6.1,12.000,17.5,18.6,22.0
4,AM,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.5,2.2,2.3,4.3,4.5,6.900,17.7,13.5,17.7
5,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,18.9,20.2,20.9,20.9,5.3,6.5,7.400,7.5,10.0,12.0
6,PA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,500.1,549.6,561.4,607.4,731.9,828.5,939.500,1129.3,1260.3,1425.4
7,TO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,964.0,988.1,1028.6,1078.0,1210.5,1340.8,1438.445,1456.7,1564.5,1678.7
8,NORDESTE,0.0,0.0,0.0,1.9,2.4,1.2,5.0,28.0,73.0,...,3095.8,3263.5,3332.2,3356.6,3544.3,3821.3,4099.200,4406.8,4679.1,5046.6
9,MA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,...,821.7,951.5,992.4,976.4,1005.7,1075.1,1192.700,1329.7,1436.1,1549.6


In [19]:
display(produtividade.head(15))

,Localidade,1976/77,1977/78,1978/79,1979/80,1980/81,1981/82,1982/83,1983/84,1984/85,...,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26 Previsão (¹)
0,NORTE,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,3060.529132,3112.433194,3091.582709,3269.841908,3164.357626,3165.816054,3313.825180,3219.919850,3541.301447,3720.651679
1,RR,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,3000.000000,3077.000000,2700.000000,3044.000000,3000.000000,3000.000000,2800.000000,3000.000000,3450.000000,3295.000000
2,RO,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,3143.000000,3283.000000,3324.000000,3541.000000,3468.000000,3105.000000,3520.000000,3579.000000,3799.000000,3895.000000
3,AC,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,0.000000,2938.000000,2940.000000,2939.000000,2688.000000,3344.000000,3808.000000,3460.000000,3459.000000,3780.000000
4,AM,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,0.000000,2250.000000,2400.000000,2300.000000,3000.000000,3000.000000,2880.000000,3060.000000,3330.000000,3060.000000
5,AP,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,2878.000000,2884.000000,2751.000000,2837.000000,2420.000000,2650.000000,2658.000000,2693.000000,2618.000000,2650.000000
6,PA,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,3270.000000,2905.000000,3044.000000,3061.000000,3048.000000,3166.000000,3090.000000,3622.000000,3568.000000,3785.000000
7,TO,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,...,2932.000000,3178.000000,3066.000000,3322.000000,3151.000000,3202.000000,3420.000000,2770.000000,3422.000000,3642.000000
8,NORDESTE,0.0,0.0,0.0,1157.894737,1583.333333,333.333333,900.0,1300.0,1158.90411,...,3115.429679,3647.346254,3311.593752,3521.296163,3626.155884,3632.578102,3848.282079,3574.929382,3743.485200,3852.429596
9,MA,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,900.00000,...,3010.000000,3180.000000,3015.000000,3206.000000,3267.000000,3276.000000,3564.000000,3251.000000,3383.000000,3528.000000


In [20]:
display(producao.head(15))

,Localidade,1976/77,1977/78,1978/79,1979/80,1980/81,1981/82,1982/83,1983/84,1984/85,...,2016/17,2017/18,2018/19,2019/20,2020/21,2021/22,2022/23,2023/24,2024/25,2025/26 Previsão (¹)
0,NORTE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5536.4972,6012.6000,6147.0,6902.1,7672.3,8779.0,10346.7,10931.4,13094.1,14960.7
1,RR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,90.0000,117.5414,108.0,151.6,210.0,285.0,344.4,369.0,468.9,479.4
2,RO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,930.3280,1095.2000,1109.2,1233.7,1375.1,1526.7,2094.4,2302.0,2639.2,2803.2
3,AC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0000,1.4690,4.4,11.8,16.4,20.4,45.7,60.6,64.3,83.2
4,AM,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0000,3.3750,5.3,5.3,12.9,13.5,19.9,54.2,45.0,54.2
5,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,54.3942,58.2568,57.5,59.3,12.8,17.2,19.7,20.2,26.2,31.8
6,PA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1635.3270,1596.5880,1708.9,1859.3,2230.8,2623.0,2903.1,4090.3,4496.8,5395.1
7,TO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2826.4480,3140.1818,3153.7,3581.1,3814.3,4293.2,4919.5,4035.1,5353.7,6113.8
8,NORDESTE,0.0,0.0,0.0,2.2,3.8,0.4,4.5,36.4,84.6,...,9644.7472,11903.1145,11034.9,11819.6,12852.2,13881.2,15774.9,15754.0,17516.1,19441.7
9,MA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,...,2473.3170,3025.7700,2992.1,3130.3,3285.6,3522.0,4250.8,4322.9,4858.3,5467.0


In [21]:
def transformar_formato_longo(df, nome_valor):
    tabela = df.copy()

    tabela = tabela.melt(
        id_vars="Localidade",
        var_name="Safra",
        value_name=nome_valor
    )

    tabela["Safra"] = tabela["Safra"].astype(str).str.strip()

    tabela[nome_valor] = pd.to_numeric(
        tabela[nome_valor],
        errors="coerce"
    )

    tabela = tabela.dropna(subset=[nome_valor])

    return tabela.reset_index(drop=True)


area_longa = transformar_formato_longo(
    area,
    "Area_mil_ha"
)

produtividade_longa = transformar_formato_longo(
    produtividade,
    "Produtividade_kg_ha"
)

producao_longa = transformar_formato_longo(
    producao,
    "Producao_mil_t"
)

In [22]:
conab = area_longa.merge(
    produtividade_longa,
    on=["Localidade", "Safra"],
    how="outer"
)

conab = conab.merge(
    producao_longa,
    on=["Localidade", "Safra"],
    how="outer"
)

conab = conab.sort_values(
    by=["Localidade", "Safra"]
).reset_index(drop=True)

display(conab.head(30))

,Localidade,Safra,Area_mil_ha,Produtividade_kg_ha,Producao_mil_t
0,AC,1976/77,0.0,0.0,0.0
1,AC,1977/78,0.0,0.0,0.0
2,AC,1978/79,0.0,0.0,0.0
3,AC,1979/80,0.0,0.0,0.0
4,AC,1980/81,0.0,0.0,0.0
5,AC,1981/82,0.0,0.0,0.0
6,AC,1982/83,0.0,0.0,0.0
7,AC,1983/84,0.0,0.0,0.0
8,AC,1984/85,0.0,0.0,0.0
9,AC,1985/86,0.0,0.0,0.0


In [23]:
brasil = conab[
    conab["Localidade"]
    .str.upper()
    .str.contains("BRASIL", na=False)
].copy()

display(brasil)

,Localidade,Safra,Area_mil_ha,Produtividade_kg_ha,Producao_mil_t
250,BRASIL,1976/77,6949.000,1747.733487,12145.00000
251,BRASIL,1977/78,7780.000,1250.128535,9726.00000
252,BRASIL,1978/79,8151.000,1251.380199,10200.00000
253,BRASIL,1979/80,8755.900,1700.270675,14887.40000
254,BRASIL,1980/81,8693.400,1781.213334,15484.80000
255,BRASIL,1981/82,8393.200,1535.874279,12890.90000
256,BRASIL,1982/83,8412.000,1727.639087,14532.90000
257,BRASIL,1983/84,9162.900,1674.197034,15340.50000
258,BRASIL,1984/85,10074.000,1807.772484,18211.50000
259,BRASIL,1985/86,9644.400,1369.447555,13207.50000


In [50]:
conab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1750 entries, 0 to 1749
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Localidade           1750 non-null   object 
 1   Safra                1750 non-null   object 
 2   Area_mil_ha          1750 non-null   float64
 3   Produtividade_kg_ha  1722 non-null   float64
 4   Producao_mil_t       1715 non-null   float64
dtypes: float64(3), object(2)
memory usage: 68.5+ KB


In [51]:
conab.shape

(1750, 5)

In [52]:
conab.isna().sum()

,0
Localidade,0
Safra,0
Area_mil_ha,0
Produtividade_kg_ha,28
Producao_mil_t,35


In [53]:
conab.describe(include="all")

,Localidade,Safra,Area_mil_ha,Produtividade_kg_ha,Producao_mil_t
count,1750,1750,1750.000000,1722.000000,1715.000000
unique,35,50,NaN,NaN,NaN
top,AC,1976/77,NaN,NaN,NaN
freq,50,35,NaN,NaN,NaN
mean,NaN,NaN,2356.770389,1678.766585,6814.906965
std,NaN,NaN,5652.419084,1341.742533,18192.941430
min,NaN,NaN,0.000000,0.000000,0.000000
25%,NaN,NaN,0.000000,0.000000,0.000000
50%,NaN,NaN,190.550000,1967.260129,479.400000
75%,NaN,NaN,1853.000000,2845.000000,4941.550000


In [73]:
conab.to_parquet(
    Path("/content/conab.parquet"),
    index=False
)

# **Dolar**

**Fonte: Banco Central do Brasil — BCB.**

**Indicador: dólar PTAX.**

**O que é: histórico diário da cotação do dólar em reais.**

**Frequência original: Diária em dias úteis.**

**O dólar é uma variável central porque a soja é uma commodity negociada internacionalmente.
Uma valorização do dólar pode:
tornar a exportação mais atrativa;
elevar o valor da soja convertido para reais;
influenciar os preços nos portos;
alterar custos de insumos importados.**

In [25]:
dolar = pd.read_csv("/content/dolar_ptax_raw.csv")
dolar.head()

,data,valor
0,2016-06-06,3.5098
1,2016-06-07,3.4745
2,2016-06-08,3.3898
3,2016-06-09,3.3817
4,2016-06-10,3.4261


In [26]:
dolar.tail()

,data,valor
2504,2026-05-28,5.0517
2505,2026-05-29,5.0569
2506,2026-06-01,5.0303
2507,2026-06-02,5.0160
2508,2026-06-03,5.0415


In [44]:
dolar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   data    2509 non-null   object 
 1   valor   2509 non-null   float64
dtypes: float64(1), object(1)
memory usage: 39.3+ KB


In [46]:
dolar.shape

(2509, 2)

In [47]:
dolar.isna().sum()

,0
data,0
valor,0


In [48]:
dolar["data"].duplicated().sum()

np.int64(0)

In [49]:
dolar.describe(include="all")

,data,valor
count,2509,2509.000000
unique,2509,NaN
top,2026-06-03,NaN
freq,1,NaN
mean,NaN,4.656981
std,NaN,0.897492
min,NaN,3.051000
25%,NaN,3.819300
50%,NaN,5.017800
75%,NaN,5.369800


In [74]:
dolar.to_parquet(
    Path("/content/dolar.parquet"),
    index=False
)

# **CEPEA**

**Fonte: CEPEA/ESALQ-USP — Centro de Estudos Avançados em Economia Aplicada.**

**Indicador: Indicador da Soja CEPEA/ESALQ — Paraná.**

**O que é: histórico diário do preço da soja no mercado físico do Paraná.**

**A unidade utilizada pelo indicador deve ser mantida conforme a documentação do CEPEA, normalmente expressa por saca de 60 kg.
Esse indicador está mais ligado ao mercado físico interno do Paraná, importante estado produtor e comercializador de soja.**

**Frequência original: diária**

=======================

**Indicador: Indicador da Soja CEPEA/ESALQ — Paranaguá.
O que é: histórico diário do preço da soja relacionado ao mercado de Paranaguá, no Paraná.**

**Por estar relacionado a Paranaguá, esse indicador possui maior ligação com:
exportação;
demanda externa;
movimentação portuária;
câmbio;
preços internacionais;
prêmio de exportação;
custos logísticos até o porto.**

**Esse será o candidato principal a target do projeto**

In [29]:
parana = pd.read_excel("/content/cepea-parana-01_16-06_26.xls")

parana.head()

,Data,À vista R$,À vista US$
0,04/01/2016,"78,14","19,30"
1,05/01/2016,"79,00","19,79"
2,06/01/2016,"78,55","19,55"
3,07/01/2016,"78,82","19,49"
4,08/01/2016,"78,96","19,57"


In [30]:
paranagua = pd.read_excel("/content/cepea-paranagua-01_16-06_26.xls")
paranagua.head()

,Data,À vista R$,À vista US$
0,04/01/2016,"81,64","20,17"
1,05/01/2016,"82,26","20,61"
2,06/01/2016,"81,84","20,37"
3,07/01/2016,"82,35","20,36"
4,08/01/2016,"83,16","20,61"


In [54]:
parana.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2597 entries, 0 to 2596
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Data         2597 non-null   object
 1   À vista R$   2597 non-null   object
 2   À vista US$  2597 non-null   object
dtypes: object(3)
memory usage: 61.0+ KB


In [55]:
parana.shape

(2597, 3)

In [56]:
parana.isna().sum()

,0
Data,0
À vista R$,0
À vista US$,0


In [57]:
parana.describe(include="all")

,Data,À vista R$,À vista US$
count,2597,2597,2597
unique,2597,2232,1311
top,02/06/2026,"77,19","19,60"
freq,1,5,10


In [58]:
paranagua.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2597 entries, 0 to 2596
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Data         2597 non-null   object
 1   À vista R$   2597 non-null   object
 2   À vista US$  2597 non-null   object
dtypes: object(3)
memory usage: 61.0+ KB


In [59]:
paranagua.shape

(2597, 3)

In [60]:
paranagua.isna().sum()

,0
Data,0
À vista R$,0
À vista US$,0


In [61]:
paranagua.describe(include="all")

,Data,À vista R$,À vista US$
count,2597,2597,2597
unique,2597,2247,1303
top,02/06/2026,"87,69","23,44"
freq,1,4,9


In [75]:
parana.to_parquet(
    Path("/content/cepea_parana.parquet"),
    index=False
)

paranagua.to_parquet(
    Path("/content/cepea_paranagua.parquet"),
    index=False
)

# **Exportações brasileiras de soja**

**Fonte original: Comex Stat, sistema de estatísticas de comércio exterior do Governo Federal.
Os dados do Comex Stat são administrados no contexto do Ministério do Desenvolvimento, Indústria, Comércio e Serviços.**

**O que é: tabela mensal consolidada das exportações brasileiras de soja.**

**volume_kg
Quantidade exportada no mês, em quilogramas.**


**volume_ton
Quantidade exportada em toneladas.**


**valor_fob_usd
Valor total exportado em dólares na condição FOB.
FOB significa que o valor considera a mercadoria colocada no local de embarque, sem incluir frete marítimo e seguro internacional pagos pelo comprador.**


**preco_implicito_usd_ton
Preço médio implícito por tonelada exportada.**

**Esse arquivo representa a demanda externa pela soja brasileira.
Frequencia original: Mensal**

In [31]:
exportacoes = pd.read_parquet("/content/fact_exports_monthly.parquet")
exportacoes.head()

,ano_mes,volume_kg,volume_ton,valor_fob_usd,preco_implicito_usd_ton
0,2016-01-01,394419947,3.944199e+05,147623117,374.279035
1,2016-02-01,2036796148,2.036796e+06,715302008,351.189788
2,2016-03-01,8373802762,8.373803e+06,2924358377,349.227043
3,2016-04-01,10085881032,1.008588e+07,3532371105,350.229305
4,2016-05-01,9915098848,9.915099e+06,3600309132,363.113791


In [66]:
exportacoes.tail()

,ano_mes,volume_kg,volume_ton,valor_fob_usd,preco_implicito_usd_ton
120,2026-01-01,1876528754,1.876529e+06,821893814,437.986262
121,2026-02-01,7101012525,7.101013e+06,2897179100,407.995211
122,2026-03-01,14518012801,1.451801e+07,5897774170,406.238392
123,2026-04-01,16756088436,1.675609e+07,6965956360,415.726880
124,2026-05-01,14825071268,1.482507e+07,6305010428,425.293769


In [62]:
exportacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ano_mes                  125 non-null    datetime64[us]
 1   volume_kg                125 non-null    int64         
 2   volume_ton               125 non-null    float64       
 3   valor_fob_usd            125 non-null    int64         
 4   preco_implicito_usd_ton  125 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(2)
memory usage: 5.0 KB


In [63]:
exportacoes.shape

(125, 5)

In [64]:
exportacoes.isna().sum()

,0
ano_mes,0
volume_kg,0
volume_ton,0
valor_fob_usd,0
preco_implicito_usd_ton,0


In [65]:
exportacoes.describe(include="all")

,ano_mes,volume_kg,volume_ton,valor_fob_usd,preco_implicito_usd_ton
count,125,1.250000e+02,1.250000e+02,1.250000e+02,125.000000
mean,2021-03-01 20:09:36,7.110063e+09,7.110063e+06,3.044005e+09,430.155362
min,2016-01-01 00:00:00,4.949834e+07,4.949834e+04,2.326652e+07,333.140857
25%,2018-08-01 00:00:00,3.292699e+09,3.292699e+06,1.498479e+09,376.624186
50%,2021-03-01 00:00:00,6.395672e+09,6.395672e+06,2.878022e+09,406.238392
75%,2023-10-01 00:00:00,1.042013e+10,1.042013e+07,4.199194e+09,467.530876
max,2026-05-01 00:00:00,1.675609e+10,1.675609e+07,8.115565e+09,631.216456
std,NaN,4.496876e+09,4.496876e+06,2.001354e+09,77.324547


In [76]:
exportacoes.to_parquet(
    Path("/content/comex_exportacoes.parquet"),
    index=False
)

# **Climaticos**

**Fonte: NASA POWER — Prediction of Worldwide Energy Resources.**

**A frequência é diária.**

**Eles não são medições diretas de uma estação meteorológica específica. São dados meteorológicos de grade produzidos a partir de modelos, reanálises e informações de satélite para as coordenadas escolhidas.**

**Cada arquivo representa um ponto geográfico próximo a uma região relevante da produção de soja.**

**T2M
Temperatura média do ar a dois metros de altura.
Unidade:
°C**

**T2M_MAX
Temperatura máxima diária a dois metros.
Unidade:
°C**

**T2M_MIN
Temperatura mínima diária a dois metros.
Unidade:
°C**

**PRECTOTCORR
Precipitação total diária corrigida.
Unidade normalmente utilizada:
mm/dia**

**RH2M
Umidade relativa do ar a dois metros de altura.
Unidade:
%**


**Luís Eduardo Magalhães está no oeste da Bahia, importante área de produção de grãos do MATOPIBA.
Rio Verde é uma das principais áreas agrícolas de Goiás, com forte produção de soja e milho.
Dourados está em uma importante área produtora de soja do Mato Grosso do Sul.
Mato Grosso é o maior produtor brasileiro de soja. Cuiabá fornece um ponto adicional para representar o clima do estado.
Sorriso está em uma das áreas mais importantes da produção de soja do Brasil.
O Paraná é um dos principais produtores de soja do Brasil. Londrina representa uma importante região agrícola do norte paranaense.
Passo Fundo está em uma importante região produtora de grãos do Rio Grande do Sul.**

In [34]:
import json

In [39]:
caminhos = {
    "luis_eduardo_magalhaes": "/content/BA_luis_eduardo_magalhaes.json",
    "rio_verde": "/content/GO_rio_verde.json",
    "dourados": "/content/MS_dourados.json",
    "cuiaba": "/content/MT_cuiaba.json",
    "sorriso": "/content/MT_sorriso.json",
    "londrina": "/content/PR_londrina.json",
    "passo_fundo": "/content/RS_passo_fundo.json"
}

In [40]:
def carregar_nasa_power(caminho, local):
    with open(caminho, "r", encoding="utf-8-sig") as arquivo:
        dados = json.load(arquivo)

    # Acessa os dados meteorológicos
    parametros = dados["properties"]["parameter"]

    # Cada parâmetro vira uma coluna
    df = pd.DataFrame(parametros)

    # A data está originalmente no índice
    df.index.name = "data"
    df = df.reset_index()

    # Converte a data
    df["data"] = pd.to_datetime(
        df["data"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

    # Identifica a cidade/local
    df.insert(0, "local", local)

    return df

In [41]:
dataframes = {}

for local, caminho in caminhos.items():
    try:
        dataframes[local] = carregar_nasa_power(
            caminho=caminho,
            local=local
        )

        print(
            f"{local}: "
            f"{dataframes[local].shape[0]} linhas e "
            f"{dataframes[local].shape[1]} colunas"
        )

    except Exception as erro:
        print(f"Erro ao carregar {local}: {erro}")

luis_eduardo_magalhaes: 3804 linhas e 7 colunas
rio_verde: 3804 linhas e 7 colunas
dourados: 3804 linhas e 7 colunas
cuiaba: 3804 linhas e 7 colunas
sorriso: 3804 linhas e 7 colunas
londrina: 3804 linhas e 7 colunas
passo_fundo: 3804 linhas e 7 colunas


In [42]:
weather = pd.concat(
    dataframes.values(),
    ignore_index=True
)

weather = weather.sort_values(
    ["local", "data"]
).reset_index(drop=True)

print("Dimensão final:", weather.shape)
display(weather.head())

Dimensão final: (26628, 7)


,local,data,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M
0,cuiaba,2016-01-01,29.03,33.61,24.42,5.60,69.86
1,cuiaba,2016-01-02,27.89,32.51,24.24,6.00,77.20
2,cuiaba,2016-01-03,26.74,29.81,24.12,11.17,83.15
3,cuiaba,2016-01-04,28.19,33.43,23.79,14.67,76.98
4,cuiaba,2016-01-05,26.80,29.98,24.46,9.06,86.03


In [43]:
print("Colunas:")
print(weather.columns.tolist())

print("\nPeríodo por local:")
display(
    weather.groupby("local")["data"]
    .agg(data_inicial="min", data_final="max", registros="count")
    .reset_index()
)

Colunas:
['local', 'data', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M']

Período por local:


,local,data_inicial,data_final,registros
0,cuiaba,2016-01-01,2026-05-31,3804
1,dourados,2016-01-01,2026-05-31,3804
2,londrina,2016-01-01,2026-05-31,3804
3,luis_eduardo_magalhaes,2016-01-01,2026-05-31,3804
4,passo_fundo,2016-01-01,2026-05-31,3804
5,rio_verde,2016-01-01,2026-05-31,3804
6,sorriso,2016-01-01,2026-05-31,3804


In [67]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26628 entries, 0 to 26627
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   local        26628 non-null  object        
 1   data         26628 non-null  datetime64[ns]
 2   T2M          26628 non-null  float64       
 3   T2M_MAX      26628 non-null  float64       
 4   T2M_MIN      26628 non-null  float64       
 5   PRECTOTCORR  26628 non-null  float64       
 6   RH2M         26628 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 1.4+ MB


In [68]:
weather.shape

(26628, 7)

In [69]:
weather.isna().sum()

,0
local,0
data,0
T2M,0
T2M_MAX,0
T2M_MIN,0
PRECTOTCORR,0
RH2M,0


In [70]:
weather.describe(include="all")

,local,data,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M
count,26628,26628,26628.000000,26628.000000,26628.000000,26628.000000,26628.000000
unique,7,NaN,NaN,NaN,NaN,NaN,NaN
top,cuiaba,NaN,NaN,NaN,NaN,NaN,NaN
freq,3804,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2021-03-16 11:59:59.999999744,24.026134,30.285588,18.580958,3.338937,67.881364
min,NaN,2016-01-01 00:00:00,0.900000,6.540000,-4.300000,0.000000,15.760000
25%,NaN,2018-08-08 18:00:00,22.250000,27.710000,16.150000,0.000000,56.150000
50%,NaN,2021-03-16 12:00:00,24.680000,30.440000,19.730000,0.300000,71.530000
75%,NaN,2023-10-23 06:00:00,26.760000,33.440000,21.870000,3.500000,81.870000
max,NaN,2026-05-31 00:00:00,37.690000,45.540000,31.820000,124.140000,97.820000


In [77]:
weather.to_parquet(
    Path("/content/weather_nasa_power.parquet"),
    index=False
)